# 02 – Spatial Join/Snap/Add to SWORD
**Purpose:** Join/ Snap and add different Dataset to the SWORD data.


**Workflow:**
1. Imports
2. ...
3. ....

---
## 2.0 | Imports

In [1]:
from shapely.geometry import Point
import geopandas as gpd
import pandas as pd
import numpy as np
import os
import sys
import fiona
import matplotlib.pyplot as plt
import contextily as ctx
import webbrowser

sys.path.append(os.path.join("..", "src"))
from config import DATA_RAW, DATA_PROCESSED, DATA_OUTPUT

SWORD = os.path.join(DATA_RAW, "0_data", "Naryn_example", "Naryn_SWORD_reaches.gpkg")

---
## 2.1 | Snap Points to Line
### 2.1.1 | Snap GDW points to SWORD lines

References: 

Ward, B. (2019): "How to leverage Geopandas for faster snapping of points to lines". URL: https://medium.com/@brendan_ward/how-to-leverage-geopandas-for-faster-snapping-of-points-to-lines-6113c94e59aa

In [33]:
# Import GlobalDamWatch (GDW) points
GDW  = os.path.join(DATA_RAW, "0_data", "Naryn_example", "Naryn_GDW_barriers_v1_0.gpkg")
# See available layers in the GeoPackage
print(fiona.listlayers(GDW))
# transforming GDW to vector data
gdw = gpd.read_file(GDW)
print(gdw.geometry.geom_type.unique())

# Inspecting the data:
print(gdw.head())
print(gdw.columns)
print(gdw.info())
print(gdw.crs)

#gdw.columns = gdw.columns.str.lower()
print(gdw.columns)

['Naryn_GDW_barriers']
<StringArray>
['Point']
Length: 1, dtype: str
   GDW_ID        RES_NAME     DAM_NAME ALT_NAME DAM_TYPE LAKE_CTRL  RIVER  \
0     287  Toktogul'skoye     Toktogul     None      Dam      None  Naryn   
1    4086             NaN    Tashkumyr     None      Dam      None  Naryn   
2    4097             NaN  Uchkurgan 1     None      Dam      None  Naryn   
3    8873             NaN          NaN     None      Dam      None    NaN   
4    8876             NaN          NaN     None      Dam      None    NaN   

  ALT_RIVER MAIN_BASIN SUB_BASIN  ... LONG_DAM LAT_DAM ORIG_SRC POLY_SRC  \
0      None       None      None  ...      0.0     0.0    GRanD     SWBD   
1      None       None      None  ...      0.0     0.0    GRanD     SWBD   
2      None       None      None  ...      0.0     0.0    GRanD     SWBD   
3      None       None      None  ...      0.0     0.0    GOODD     SWBD   
4      None       None      None  ...      0.0     0.0    GOODD     SWBD   

  GRAND_ID 

In [43]:
# Filtering Instream vs. Offstrem Features
# Checking fo individual Values in certain column:
gdw["INSTREAM"].unique()
gdw = gdw[gdw["INSTREAM"].str.lower() == "instream"]

gdw["INSTREAM"].unique()

<StringArray>
['Instream']
Length: 1, dtype: str

In [49]:
# Transform SWORD to vectorized data:
sword = gpd.read_file(SWORD)
print(sword.geometry.geom_type.unique())

# transform the data to a CRS in meters:
print(gdw.crs, sword.crs)
gdw = gdw.to_crs("EPSG:32643")
sword = sword.to_crs("EPSG:32643")
print(gdw.crs, sword.crs)

<StringArray>
['MultiLineString']
Length: 1, dtype: str
EPSG:4326 EPSG:4326
EPSG:32643 EPSG:32643


In [63]:
# creating spatial index on sword to select only features which overlap with gdw bounding box:
sword.sindex

# create bounding boxes or gdw:
offset = 100
gdw_bbox = gdw.bounds + [-offset, -offset, offset, offset]

hits = gdw_bbox.apply(lambda row: list(sword.sindex.intersection(row)), axis=1)
print(hits)

# Check if one gdw point has no spatial matching line:
has_match = hits.apply(len) > 0
print("All gdw points have a sword line match: ", has_match.all())
# Check which point has no line:
gdw_no_match = gdw[~has_match]
print("No match:", gdw_no_match)

0        [32]
1         [8]
2         [3]
3        [16]
4         [6]
5        [79]
6    [23, 22]
dtype: object
All gdw points have a sword line match:  True
No match: Empty GeoDataFrame
Columns: [GDW_ID, RES_NAME, DAM_NAME, ALT_NAME, DAM_TYPE, LAKE_CTRL, RIVER, ALT_RIVER, MAIN_BASIN, SUB_BASIN, COUNTRY, SEC_CNTRY, ADMIN_UNIT, SEC_ADMIN, NEAR_CITY, ALT_CITY, YEAR_DAM, PRE_YEAR, YEAR_SRC, ALT_YEAR, REM_YEAR, TIMELINE, YEAR_TXT, DAM_HGT_M, ALT_HGT_M, DAM_LEN_M, ALT_LEN_M, AREA_SKM, AREA_POLY, AREA_REP, AREA_MAX, AREA_MIN, CAP_MCM, CAP_MAX, CAP_REP, CAP_MIN, DEPTH_M, DIS_AVG_LS, DOR_PC, ELEV_MASL, CATCH_SKM, CATCH_REP, POWER_MW, DATA_INFO, USE_IRRI, USE_ELEC, USE_SUPP, USE_FCON, USE_RECR, USE_NAVI, USE_FISH, USE_PCON, USE_LIVE, USE_OTHR, MAIN_USE, MULTI_DAMS, COMMENTS, URL, QUALITY, EDITOR, LONG_RIV, LAT_RIV, LONG_DAM, LAT_DAM, ORIG_SRC, POLY_SRC, GRAND_ID, HYRIV_ID, INSTREAM, HYLAK_ID, HYBAS_L12, geometry]
Index: []

[0 rows x 72 columns]


In [66]:
# create a longer DataFrame with one row for each element in the list of line indexes:
tmp = pd.DataFrame({
    # index of points table
    "gdw_idx": np.repeat(hits.index, hits.apply(len)),
    # ordinal position of line - access via iloc later
    "sword_i": np.concatenate(hits.values)
})

# Join back to the sword on sword_i; we use reset_index() to give us the ordinal position of each line:
tmp = tmp.join(sword.reset_index(drop=True), on="sword_i")

# Join back to the original gdws to get their geometry rename the gdw geometry as "point":
tmp = tmp.join(gdw.geometry.rename("point"), on="gdw_idx")

# Convert back to a GeoDataFrame, so we can do spatial ops
tmp = gpd.GeoDataFrame(tmp, geometry="geometry", crs=gdw.crs)

Find the closest sword line to each gdw point:

In [ ]:
tmp["snap_dist"] = tmp.geometry.distance(gpd.GeoSeries(tmp.point))

# Discard any sword lines that are greater than tolerance from gdw pints:
# Print mean, median and biggest distance for all calculated distances:
# print("All calculated distances:\n",
#     "Mean:", tmp["snap_dist"].mean(),
#       "|Median:", tmp["snap_dist"].median(),
#       "|Max:", tmp["snap_dist"].max())
print(tmp["snap_dist"].describe())

# Identify GDW points that matched more than one SWORD line
match_counts = tmp.groupby("gdw_idx")["sword_i"].transform("count")

# Keep only rows where the GDW point has multiple SWORD matches
tmp_multiple = tmp[match_counts > 1]
# Print mean, median and biggest distance from all gdw points with multiple sword line matches:
print(tmp_multiple["snap_dist"].describe())

All calculated distances:
 Mean: 91.43348162397848 |Median: 81.18853801097941 |Max: 182.90673046873775
Distances for GDW points with multiple SWORD matches:
 Mean: 163.058398992171 | Median: 163.058398992171 | Max: 182.90673046873775


In [79]:
# Discarding any sword lines that are greater than tolerance from gdw points:
tolerance = 164
tmp = tmp.loc[tmp.snap_dist <= tolerance]

# Sort on ascending snap distance, so that closest goes to top
tmp = tmp.sort_values(by=["snap_dist"])

# group by the index of the gdw points and take the first, which is the closest line 
closest = tmp.groupby("gdw_idx").first()
# construct a GeoDataFrame of the closest lines
closest = gpd.GeoDataFrame(closest, geometry="geometry")

Find the point on the line that is closest to the original point:

In [ ]:
# Position of nearest gdw point from start of the sword line
pos = closest.geometry.project(gpd.GeoSeries(closest.point))
# Get new point location geometry
snapped_pts = closest.geometry.interpolate(pos)

#Identify the columns we want to copy from the closest line to the point, such as a line ID.
print(closest.columns)
line_columns = ["sword_i", "reach_id", "network"]
# Create a new GeoDataFrame from the columns from the closest line and new point geometries (which will be called "geometries")
snapped = gpd.GeoDataFrame(
closest[line_columns], geometry = snapped_pts)

# Join back to the original points:
updated_points = gdw.drop(columns=["geometry"]).join(snapped)
# You may want to drop any that didn't snap, if so:
updated_points = updated_points.dropna(subset=["geometry"])

Index(['sword_i', 'x', 'y', 'reach_id', 'reach_len', 'n_nodes', 'wse',
       'wse_var', 'width', 'width_var', 'facc', 'n_chan_max', 'n_chan_mod',
       'obstr_type', 'grod_id', 'hfalls_id', 'slope', 'dist_out', 'lakeflag',
       'max_width', 'n_rch_up', 'n_rch_dn', 'rch_id_up', 'rch_id_dn',
       'swot_orbit', 'swot_obs', 'type', 'river_name', 'edit_flag',
       'trib_flag', 'path_freq', 'path_order', 'path_segs', 'main_side',
       'strm_order', 'end_reach', 'network', 'geometry', 'point', 'snap_dist'],
      dtype='str')


Export snapped GDW Points:

In [ ]:
# Export snapped gdw points as gpkg:
# Making sure gpd is a GeoDataFrame:
updated_points = gpd.GeoDataFrame(updated_points, geometry="geometry", crs=gdw.crs)

#output_path = os.path.join(DATA_PROCESSED, "snapped_gdw_points.gpkg")

updated_points.to_file(
    os.path.join(DATA_PROCESSED, "snapped_Naryn_GDW_barriers_v1_0.gpkg"),
    layer="snapped_Naryn_GDW_barriers_v1_0",
    driver="GPKG"
)

---
## 2.2 | Spatial Join
### 2.2.1 | Join GRITv1.0 strahler order information to SWORD

What I want to do:

I want to define a boundary for the egg that includes all the reaches within it. To do this, I want to use the strahler order, which I think works pretty well based on the GRITv1.0 dataset. This means that a single strahler order will contain multiple SWORD reaches. I then want to create one egg per strahler. 

To-Do:
- [ ] How should I handle reaches which falls within the coverage of multiple strahler? 
- [ ]

In [20]:
# ============================================================
# CONFIGURATION – only edit this cell
# ============================================================
# Inputs:
SWORD_CLIPPED_PATH = os.path.join(DATA_PROCESSED, "sword_naryn_clip.gpkg")
GRIT_PATH  = os.path.join(DATA_RAW, "0_data", "Naryn_example", "Naryn_GRITv1.0.gpkg")
GRIT_LAYER = None
# ============================================================

# Columns to take from GRIT:
GRIT_COLS_TO_JOIN = [
    "strahler_order",  # main information
    "global_id",       # provenance reference
    "geometry"         # required for spatial join
]

# How to Rename columns after join (Format: {"original_name": "new_name"}):
GRIT_RENAME = {
    "global_id": "global_id_GRITv1.0",
    "strahler_order": "strahler_order_GRITv1.0"
}

# tolerance for geometry mismatch between datasets:
MAX_DISTANCE_METERS = 100  

# Outputs:
OUT_JOINED = os.path.join(DATA_PROCESSED, "sword_naryn_GRITv1.0_joined.gpkg")
OUT_MAP    = os.path.join(DATA_OUTPUT,    "02_join_GRITv1.0_map.html")

Load Data:

In [6]:
# Load SWORD:
sword = gpd.read_file(SWORD_CLIPPED_PATH)
print(f"SWORD reaches loaded : {len(sword)}")
print(f"SWORD CRS: {sword.crs}")
print(f"GRIT columns: {sword.columns.tolist()}")

# Load GRIT
grit_layers = fiona.listlayers(GRIT_PATH)
grit_layer  = GRIT_LAYER if GRIT_LAYER else grit_layers[0]
grit_raw = gpd.read_file(GRIT_PATH, layer=grit_layer)

print(f"\nGRIT features loaded : {len(grit_raw)}")
print(f"GRIT CRS: {grit_raw.crs}")
print(f"GRIT columns: {grit_raw.columns.tolist()}")

SWORD reaches loaded : 140
SWORD CRS: EPSG:4326
GRIT columns: ['x', 'y', 'reach_id', 'reach_len', 'n_nodes', 'wse', 'wse_var', 'width', 'width_var', 'facc', 'n_chan_max', 'n_chan_mod', 'obstr_type', 'grod_id', 'hfalls_id', 'slope', 'dist_out', 'lakeflag', 'max_width', 'n_rch_up', 'n_rch_dn', 'rch_id_up', 'rch_id_dn', 'swot_orbit', 'swot_obs', 'type', 'river_name', 'edit_flag', 'trib_flag', 'path_freq', 'path_order', 'path_segs', 'main_side', 'strm_order', 'end_reach', 'network', 'geometry']

GRIT features loaded : 710
GRIT CRS: EPSG:4326
GRIT columns: ['cat', 'global_id', 'catchment_id', 'upstream_node_id', 'downstream_node_id', 'upstream_line_ids', 'downstream_line_ids', 'direction_algorithm', 'width_adjusted', 'length_adjusted', 'is_mainstem', 'strahler_order', 'cycle', 'length', 'azimuth', 'sinuousity', 'drainage_area_in', 'drainage_area_out', 'drainage_area_mainstem_in', 'drainage_area_mainstem_out', 'bifurcation_balance_out', 'grwl_overlap', 'grwl_value', 'name', 'name_local', 'n_


Prepare GRIT

In [7]:
# Only keeping the needed columns:
# Check that all requested columns actually exist in GRIT:
missing_cols = [c for c in GRIT_COLS_TO_JOIN if c != "geometry"
                and c not in grit_raw.columns]
if missing_cols:
    print(f"Columns not found in GRIT: {missing_cols}")
    print("Check column names in GRIT and update GRIT_COLS_TO_JOIN")
else:
    print("All requested columns found in GRIT")

# Selecting wanted column:
cols_to_keep = [c for c in GRIT_COLS_TO_JOIN if c in grit_raw.columns]
grit = grit_raw[cols_to_keep].copy()

print(f"\nGRIT prepared: {len(grit)} features, columns: {grit.columns.tolist()}")

All requested columns found in GRIT

GRIT prepared: 710 features, columns: ['strahler_order', 'global_id', 'geometry']


In [10]:
# transform the data to a CRS in meters:
print(grit.crs, sword.crs)
grit = grit.to_crs("EPSG:32643")
sword = sword.to_crs("EPSG:32643")
print(grit.crs, sword.crs)

EPSG:4326 EPSG:32643
EPSG:32643 EPSG:32643


SPATIAL JOIN

Creating points along the SWORD reaches and assigning each point to its nearest GRIT feature. By majority vote, the strahler order of GRIT is choosen for each reach.
The problem with sjoin_nearest was, that only compared thenearest point between two geometries which lead to missclassification.

In [23]:
# sample a point every 50 meters along each reach:
SAMPLE_DISTANCE_M = 50

# For each reach, calculate how many sample points needed:
reach_lengths = sword.geometry.length
n_samples = (reach_lengths / SAMPLE_DISTANCE_M).astype(int).clip(lower=1)

# Build arrays of reach indices and distances vectorized
reach_indices = np.repeat(sword.index.values, n_samples.values)
distances     = np.concatenate([
    np.linspace(0, length, n)
    for length, n in zip(reach_lengths.values, n_samples.values)
])

# Interpolate points along each reach geometry
reach_geoms_repeated = sword.geometry.loc[reach_indices]
sample_points = reach_geoms_repeated.interpolate(distances)

# Build a GeoDataFrame of all sample points
samples = gpd.GeoDataFrame(
    {"sword_index": reach_indices},
    geometry=sample_points.values,
    crs=sword.crs
)

print(f"  SWORD reaches     : {len(sword)}")
print(f"  Sample points     : {len(samples)}")
print(f"  Avg points/reach  : {len(samples)/len(sword):.1f}")

  SWORD reaches     : 140
  Sample points     : 29681
  Avg points/reach  : 212.0


Assign majority strahler order:

For each reach, count how many sample points were assigned to each strahler_order → the most frequent one wins. In case of a tie, the lower strahler order wins (more conservative).

In [33]:
#Assign each sample point to nearest GRIT feature:
points_joined = gpd.sjoin_nearest(
    samples,
    grit[["strahler_order", "global_id", "geometry"]],
    how="left",
    max_distance=MAX_DISTANCE_METERS,
    #distance_col="dist_to_grit_m"
)

# Majority vote per SWORD reach:
def majority_vote(series):
    """Return the most frequent value. Ties go to the lower value."""
    return series.value_counts().idxmax()

# Group by sword reach index and take majority strahler_order
majority = (points_joined
    .dropna(subset=["strahler_order"])           # ignore unmatched points
    .groupby("sword_index")["strahler_order"]
    .agg(majority_vote)
    .rename("strahler_order_majority"))

# Also take the global_id of the majority-winning GRIT feature:
majority_id = (points_joined
    .dropna(subset=["strahler_order"])
    .groupby("sword_index")
    .apply(lambda df: df.loc[
        df["strahler_order"] == majority_vote(df["strahler_order"]),
        "global_id"
    ].iloc[0])
    .rename("global_id_majority"))

# Join majority result back to SWORD:
sword_joined = sword.join(majority).join(majority_id)

# Rename to final column names
sword_joined = sword_joined.rename(columns={
    "strahler_order_majority": "strahler_order_GRITv1.0",
    "global_id_majority"     : "global_id_GRITv1.0"
})

# Quality report
n_matched   = sword_joined["strahler_order_GRITv1.0"].notna().sum()
n_unmatched = sword_joined["strahler_order_GRITv1.0"].isna().sum()

print(f"\nRows total: {len(sword_joined)}")
print(f"Rows with match: {n_matched}")
print(f"Rows without match: {n_unmatched}")
print(f"\nReaches per strahler_order_GRITv1.0:")
print(sword_joined["strahler_order_GRITv1.0"].value_counts().sort_index())

#Add assignment confidence score:
confidence = (points_joined
    .dropna(subset=["strahler_order"])
    .groupby("sword_index")
    .apply(lambda df: (
        df["strahler_order"] == majority_vote(df["strahler_order"])
    ).mean())
    .rename("grit_majority_confidence"))

sword_joined = sword_joined.join(confidence)

print("\nConfidence score distribution:")
print(sword_joined["grit_majority_confidence"].describe())


Rows total: 140
Rows with match: 138
Rows without match: 2

Reaches per strahler_order_GRITv1.0:
strahler_order_GRITv1.0
1.0       8
2.0       3
3.0      10
4.0       5
5.0       5
         ..
107.0     2
108.0     2
110.0     1
112.0     2
113.0     1
Name: count, Length: 68, dtype: int64

Confidence score distribution:
count    138.000000
mean       0.761117
std        0.234080
min        0.206897
25%        0.552872
50%        0.780369
75%        1.000000
max        1.000000
Name: grit_majority_confidence, dtype: float64


Clean Table a little bit:

In [34]:
# Drop the auto-generated index column from sjoin
sword_joined = sword_joined.drop(columns=["index_right"], errors="ignore")

# Rename columns
sword_joined = sword_joined.rename(columns=GRIT_RENAME)

print("Columns after join and rename:")
print(sword_joined.columns.tolist())

# Quick check: what do the new columns look like?
# Quick check: what do the new columns look like?
sword_joined[["reach_id", "strahler_order_GRITv1.0",
              "global_id_GRITv1.0", 
              "grit_majority_confidence"]].head(10)

Columns after join and rename:
['x', 'y', 'reach_id', 'reach_len', 'n_nodes', 'wse', 'wse_var', 'width', 'width_var', 'facc', 'n_chan_max', 'n_chan_mod', 'obstr_type', 'grod_id', 'hfalls_id', 'slope', 'dist_out', 'lakeflag', 'max_width', 'n_rch_up', 'n_rch_dn', 'rch_id_up', 'rch_id_dn', 'swot_orbit', 'swot_obs', 'type', 'river_name', 'edit_flag', 'trib_flag', 'path_freq', 'path_order', 'path_segs', 'main_side', 'strm_order', 'end_reach', 'network', 'geometry', 'strahler_order_GRITv1.0', 'global_id_GRITv1.0', 'grit_majority_confidence']


,reach_id,strahler_order_GRITv1.0,global_id_GRITv1.0,grit_majority_confidence
0,46219400056,16.0,30075767.0,1.000000
1,46219400041,19.0,30075753.0,0.332461
2,46219400021,23.0,30079274.0,0.415459
3,46219300051,68.0,30075709.0,0.339806
4,46219300061,65.0,30075696.0,1.000000
5,46219300091,62.0,30075692.0,1.000000
6,46219100011,113.0,30075917.0,0.662614
7,46219100021,112.0,30075839.0,1.000000
8,46219100031,112.0,30075839.0,0.873418
9,46219100041,110.0,30075814.0,0.427966


In [35]:
# Convert to integer:
INT_COLS = ["strahler_order_GRITv1.0", "global_id_GRITv1.0"]

for col in INT_COLS:
    if col in sword_joined.columns:
        sword_joined[col] = sword_joined[col].astype("Int64")
        print(f"Converted to Int64: {col}")
    else:
        print(f"Column not found, skipping: {col}")

# Verify
print("\nData types after conversion:")
print(sword_joined[INT_COLS].dtypes)
print("\nSample:")
print(sword_joined[["reach_id"] + INT_COLS].head())

Converted to Int64: strahler_order_GRITv1.0
Converted to Int64: global_id_GRITv1.0

Data types after conversion:
strahler_order_GRITv1.0    Int64
global_id_GRITv1.0         Int64
dtype: object

Sample:
      reach_id  strahler_order_GRITv1.0  global_id_GRITv1.0
0  46219400056                       16            30075767
1  46219400041                       19            30075753
2  46219400021                       23            30079274
3  46219300051                       68            30075709
4  46219300061                       65            30075696


Checking how clearly a SWORD reahc has been assigned to a strahler by grit_majority_confidence. "1" means that all points (100%) of a SWORD habe been assigned to one strahler.

In [36]:
print("Majority confidence score [0–1]:")
print(sword_joined["grit_majority_confidence"].describe())

# Flag ambiguous reaches (confidence below 0.6)
ambiguous = sword_joined["grit_majority_confidence"] < 0.6
print(f"\nAmbiguous reaches (confidence < 0.6): {ambiguous.sum()}")

print("\nReaches per strahler_order_GRITv1.0:")
print(sword_joined["strahler_order_GRITv1.0"].value_counts().sort_index())

Majority confidence score [0–1]:
count    138.000000
mean       0.761117
std        0.234080
min        0.206897
25%        0.552872
50%        0.780369
75%        1.000000
max        1.000000
Name: grit_majority_confidence, dtype: float64

Ambiguous reaches (confidence < 0.6): 45

Reaches per strahler_order_GRITv1.0:
strahler_order_GRITv1.0
1       8
2       3
3      10
4       5
5       5
       ..
107     2
108     2
110     1
112     2
113     1
Name: count, Length: 68, dtype: Int64


Export:

In [37]:
os.makedirs(DATA_PROCESSED, exist_ok=True)
sword_joined.to_file(OUT_JOINED, driver="GPKG")
print(f"Saved: {OUT_JOINED}")

Saved: C:\Users\Duck\Documents\Studium\EAGLE\04_semester\0_InnoLab\RiverEggCode\data\processed\sword_naryn_GRITv1.0_joined.gpkg


INTERACTIVE MAP – color reaches by strahler_order_GRITv1.0

In [38]:
m = sword_joined.explore(
    column="strahler_order_GRITv1.0",
    cmap="RdYlBu",
    tooltip=["reach_id", "river_name", "strahler_order_GRITv1.0",
             "strm_order", "grit_majority_confidence"],
    tiles="CartoDB positron",
    legend=True
)

m.save(OUT_MAP)
webbrowser.open(OUT_MAP)
print(f"Map saved: {OUT_MAP}")

Map saved: C:\Users\Duck\Documents\Studium\EAGLE\04_semester\0_InnoLab\RiverEggCode\data\outputs\02_join_GRITv1.0_map.html
